# Book Market Analytics — Data Science 500

This notebook documents the project from data acquisition to interpretation. Run the cells in order. The scraper needs an internet connection.


## 1. Setup

Install dependencies from `requirements.txt` in the PyCharm project interpreter before running this notebook.


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
# If the notebook was launched from the notebooks folder, move to the project root.
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)


## 2. Data acquisition

BeautifulSoup reads each product listing. The scraper follows pagination until there are no more pages. This cell can take a short time and requires internet access.


In [ ]:
from src.scraper import scrape_catalogue

raw_df = scrape_catalogue()
raw_path = ROOT / 'data' / 'raw' / 'books_raw.csv'
raw_path.parent.mkdir(parents=True, exist_ok=True)
raw_df.to_csv(raw_path, index=False)
print(f'Scraped {len(raw_df)} unique books.')
display(raw_df.head())


## 3. Data cleaning

Convert fields to numeric values, standardise text, and remove rows that cannot be used for price modelling. Missing stock quantity is retained as missing rather than guessed.


In [ ]:
from src.analysis import clean_data

clean_df = clean_data(raw_df)
clean_path = ROOT / 'data' / 'processed' / 'books_clean.csv'
clean_path.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(clean_path, index=False)
print('Rows before cleaning:', len(raw_df))
print('Rows after cleaning:', len(clean_df))
print('Missing values by column:')
display(clean_df.isna().sum().to_frame('missing_count'))
display(clean_df.head())


## 4. Descriptive statistics


In [ ]:
display(clean_df[['price_gbp', 'rating', 'stock_quantity']].describe())
print('\nStock availability counts:')
display(clean_df['stock_status'].value_counts(dropna=False).to_frame('books'))


## 5. Visualisation

These plots examine the price distribution, price variation by rating, and stock availability. If you add category collection, a category chart can also be generated.


In [ ]:
from src.analysis import make_visualisations

figure_paths = make_visualisations(clean_df)
for figure_path in figure_paths:
    print(figure_path)
    display(plt.imread(figure_path))


## 6. Linear regression model

The model predicts price using rating, stock status, and stock quantity. An 80/20 train-test split estimates performance on held-out records. MAE and RMSE are measured in GBP; R² is a relative fit measure.


In [ ]:
from src.analysis import fit_price_model

model, metrics = fit_price_model(clean_df)
print(json.dumps(metrics, indent=2))
metrics_path = ROOT / 'reports' / 'model_metrics.json'
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')


## 7. Interpretation and limitations

Use the values and charts produced above to write the final interpretation. A low or negative R² means the selected features do not explain test-set price variation well. This baseline does not establish causation. The catalogue listing page does not expose genre for each book, so category-level conclusions require extending the scraper to visit product/category pages.

### Report prompts
- How many books were collected and retained after cleaning?
- What are the mean and median prices?
- Do prices appear concentrated or skewed?
- How does price vary across ratings, and how much do groups overlap?
- What do MAE, RMSE, and R² say about model usefulness?
- What additional features could improve the model?
